# Absorption lock-in scan analysis

Analyze a **range of exp12 shots** that scan one runmanager global (for example `mod_offset`), using the same `fluorescence_absorption_lockin.py` helper as the single-shot notebooks.

**To use:** set `DAY_DIR`, `FIRST_SHOT` and `LAST_SHOT` in the next cell, then **Restart Kernel and Run All**. Every `.h5` file in the shot folders from `FIRST_SHOT` to `LAST_SHOT` is analyzed, so a whole runmanager sequence in one folder (files `_00`, `_01`, …) needs `FIRST_SHOT = LAST_SHOT` = that folder number. The scan variable is found automatically as the global that changes across the shots (`rep` and `t_PMT_delay` are ignored); set `SCAN_GLOBAL` to override.

For each shot the notebook decides the pass from its own `t_PMT_delay` (0.016 s upward, 0.484 s returning). It then computes the 1f and 2f lock-in peaks, a **signed** 1f signal, the SNR, the PMT contrast, the MOT atom signal, and the PDA DC level:

- **SNR** uses a background more than 20 ms from the cloud (cloud ±8 ms plus a 12 ms guard). On 2026-10-14 this matched the no-atom noise to within 5%, whereas the old ±8 ms window read about 20% high.
- **Signed 1f** is the background-subtracted peak projected onto one reference phase per pass: the phase of the strongest shots in the scan. It changes sign across line center, whereas R = |Z| cannot.
- A pure-noise shot gives an SNR of about 2.4, which is the single-shot false-detection level.
- Shots with `MRR_TRIG_BOOL = 0` (no launch) are flagged, since they contain no atom signal.

Outputs (tables, plots, per-shot CSV) go to `DAY_DIR/scan_analysis_outputs/<first>-<last>/`. HDF5 files are opened read-only.

In [ ]:
# Scan analysis: imports, inputs, and plotting helpers shared with the single-shot notebooks.
import sys
import importlib
import warnings
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import display, Markdown

# CHANGE THESE INPUTS for another scan.
DAY_DIR = Path(r"C:\Experiments\eedmLabscriptOps\exp12_switchArmsAndLaunch_Absorption\2026\10\14")
FIRST_SHOT, LAST_SHOT = 49, 68
SCAN_GLOBAL = None           # None = auto-detect the global that changes across the shots
# Not scan variables: repeat counters, and t_PMT_delay (the upward/return pass, already split by _0/_1).
IGNORE_GLOBALS = {"rep", "t_PMT_delay"}

# Lock-in settings, identical to the single-shot notebooks except the wider background guard.
AVERAGING_PERIODS = 2
CLOUD_HALF_WIDTH_S = 0.008
BACKGROUND_GUARD_S = 0.012   # background > 20 ms from the cloud, where its wings no longer leak in
SWITCH_GUARD_S = 0.001

# Import the shared helper from this notebook's folder and refresh it on Run All.
import fluorescence_absorption_lockin as lockin_helper
importlib.reload(lockin_helper)
from fluorescence_absorption_lockin import (load_shot, fit_modulation_reference, analyze_shot,
                                            load_mot_trace, fit_mot_loading)

SAVE_OUTPUTS = True
OUTPUT_DIR = DAY_DIR / "scan_analysis_outputs" / f"{FIRST_SHOT:04d}-{LAST_SHOT:04d}"
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

%matplotlib widget
plt.ioff()
plt.rcParams.update({"figure.dpi": 110, "font.size": 12, "axes.grid": True, "grid.alpha": 0.25,
                     "lines.linewidth": 1.0, "axes.labelsize": 12, "axes.titlesize": 12,
                     "legend.fontsize": 11, "xtick.labelsize": 12, "ytick.labelsize": 12})
label_font = {"fontname": "sans-serif", "size": 12, "color": "black", "weight": "normal",
              "verticalalignment": "bottom"}
title_font = {"fontname": "sans-serif", "size": 12, "color": "black", "weight": "normal"}
legend_font = {"family": "sans-serif", "size": 11, "style": "normal", "weight": "normal"}
EXPORT_SIZES_INCHES = {"scan_upward": (16, 11), "scan_returning": (16, 11)}

def show_figure(fig, name):
    """Export at a fixed size and display the live, resizable widget canvas."""
    if SAVE_OUTPUTS:
        display_size = fig.get_size_inches().copy()
        try:
            fig.set_size_inches(EXPORT_SIZES_INCHES[name], forward=False)
            fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=180, bbox_inches="tight")
        finally:
            fig.set_size_inches(display_size, forward=False)
    fig.canvas.layout.width = "100%"
    fig.canvas.layout.height = "auto"
    fig.canvas.resizable = True
    fig.canvas.toolbar_visible = True
    fig.canvas.toolbar_position = "top"
    fig.canvas.header_visible = False
    display(fig.canvas)

files = sorted(p for n in range(FIRST_SHOT, LAST_SHOT + 1)
               for p in (DAY_DIR / f"{n:04d}").glob("*.h5"))
print(f"{len(files)} files for shots {FIRST_SHOT:04d}–{LAST_SHOT:04d} in {DAY_DIR}")

## Per-shot analysis

Each file is analyzed independently. Files that have not run yet (no traces) or fail to load are listed and skipped rather than stopping the scan.

In [ ]:
# Analyze every file; keep the complex peak so a common reference phase can be applied later.
warnings.filterwarnings("ignore", category=RuntimeWarning)
OPTIONS = dict(n_periods=AVERAGING_PERIODS, cloud_half_width_s=CLOUD_HALF_WIDTH_S,
               background_guard_s=BACKGROUND_GUARD_S, switch_guard_s=SWITCH_GUARD_S, harmonics=(1, 2))
RETURN_DELAY_S = 0.2   # t_PMT_delay above this is the returning pass (0.484 s); below, upward (0.016 s)
# A loaded MOT gives about 42 mV of atom signal (70 dB); a repump dropout leaves under 1 mV.
MIN_MOT_ATOMS_MV = 10.0
# The PDA100A2 output saturates near 10.2 V; any sample at or above this marks a clipped record.
CLIP_V = 9.95
EXCLUDE_FLAGGED = True  # drop no-MOT, no-launch, and clipped shots from the summary and plots

def numeric_globals(path):
    """All scalar numeric globals of a shot, for grouping and scan detection."""
    with h5py.File(path, "r") as h5:
        out = {}
        for key, value in h5["globals"].attrs.items():
            if np.ndim(value) == 0 and isinstance(value, (int, float, np.integer, np.floating, np.bool_)):
                out[key] = float(value)
        return out

rows, skipped = [], []
for path in files:
    try:
        shot = load_shot(path)
        reference = fit_modulation_reference(shot, frequency_guess_hz=shot["mod_freq"] or 6123.0)
        result = analyze_shot(shot, reference, **OPTIONS)
    except Exception as error:
        skipped.append({"File": path.name, "Reason": f"{type(error).__name__}: {error}"})
        continue
    mot = load_mot_trace(path)
    mot_fit = fit_mot_loading(mot) if mot is not None else None
    t, v = shot["t_pda"], shot["pda_v"]
    enabled = (t >= shot["enable_start_s"] + 0.002) & (t <= shot["enable_stop_s"] - 0.002)
    # A runmanager sequence puts every shot in one folder (_00, _01, ...), so the pass comes from
    # the shot's own t_PMT_delay rather than from the filename.
    row = {"file": path.name, "sequence": path.parent.name, "run": path.stem.split("_")[-1],
           "acq": "1" if shot["pmt_delay_s"] > RETURN_DELAY_S else "0",
           "PDA_V": float(v[enabled].mean()), "PDA_max_V": float(v.max()),
           "MOT_atoms_mV": 1e3 * mot_fit["atoms_v_at_coils_off"] if mot_fit else np.nan,
           "PMT_SNR": result["results"][("PMT", 1)]["peak_amp"] / result["results"][("PMT", 1)]["noise_rms"],
           "PDA_peak_t_ms": 1e3 * (result["results"][("PDA100A2", 1)]["peak_time_s"]
                                   + shot["acquisition_after_drop_s"])}
    for h in (1, 2):
        r = result["results"][("PDA100A2", h)]
        row[f"peak{h}_mV"] = 1e3 * r["peak_amp"]
        row[f"noise{h}_mV"] = 1e3 * r["noise_rms"]
        row[f"SNR{h}"] = r["peak_amp"] / r["noise_rms"]
        row[f"z{h}"] = complex(r["z"][r["peak_index"]])     # background-subtracted complex peak
    row.update(numeric_globals(path))
    rows.append(row)
    print(f"{path.name}: 1f SNR {row['SNR1']:.1f}", end="\r")

shots = pd.DataFrame(rows)
print(f"\nAnalyzed {len(shots)} files ({(shots['acq'] == '0').sum()} upward, "
      f"{(shots['acq'] == '1').sum()} returning); skipped {len(skipped)}.")
if skipped:
    display(pd.DataFrame(skipped).set_index("File"))

# Flag shots that cannot carry a clean atom signal, so a hardware problem is not mistaken for a scan result.
shots["flag"] = ""
def add_flag(mask, text):
    shots.loc[mask, "flag"] = (shots.loc[mask, "flag"] + " " + text).str.strip()
if "MRR_TRIG_BOOL" in shots:
    add_flag(shots["MRR_TRIG_BOOL"] == 0, "no launch")
add_flag(shots["MOT_atoms_mV"] < MIN_MOT_ATOMS_MV, "no MOT")
add_flag(shots["PDA_max_V"] >= CLIP_V, "clipped")
flagged = shots[shots["flag"] != ""]
if not flagged.empty:
    display(Markdown(f"**Warning: {len(flagged)} of {len(shots)} shots flagged** "
                     f"(`no MOT` = MOT atom signal below {MIN_MOT_ATOMS_MV:g} mV, e.g. repump unlocked; "
                     "`no launch` = `MRR_TRIG_BOOL = 0`; "
                     f"`clipped` = PDA reached {CLIP_V:g} V, its output limit)."
                     + (" They are excluded below; set `EXCLUDE_FLAGGED = False` to keep them."
                        if EXCLUDE_FLAGGED else "")))
    display(flagged[["file", "acq", "flag", "MOT_atoms_mV", "PDA_max_V", "PMT_SNR", "SNR1"]]
            .set_index("file").round(2))
    if EXCLUDE_FLAGGED:
        shots = shots[shots["flag"] == ""].reset_index(drop=True)
        if shots.empty:
            raise ValueError("Every shot is flagged; set EXCLUDE_FLAGGED = False to inspect them anyway")

## Scan variable and signed signal

The scan variable is the global with more than one value across the analyzed shots, excluding `IGNORE_GLOBALS`. The reference phase for each acquisition and harmonic is the circular mean phase of the strongest quarter of shots. Projecting each shot's complex peak onto that phase gives a signed amplitude that crosses zero at line center.

In [ ]:
# Identify the scan variable and project each peak onto a common phase per acquisition.
global_columns = [c for c in shots.columns
                  if c not in IGNORE_GLOBALS and c not in ("shot",) and shots[c].dtype != object
                  and c in numeric_globals(files[0])]
varying = [c for c in global_columns if shots[c].nunique() > 1]
scan = SCAN_GLOBAL or (varying[0] if varying else None)
if scan is None:
    raise ValueError("No global changes across these shots; set SCAN_GLOBAL or widen the shot range")
if len(varying) > 1 and SCAN_GLOBAL is None:
    display(Markdown(f"**Several globals vary:** {varying}. Using `{scan}`; set `SCAN_GLOBAL` to choose."))

for acq, group in shots.groupby("acq"):
    for h in (1, 2):
        strongest = group.nlargest(max(1, len(group) // 4), f"peak{h}_mV")
        phase = np.angle(np.mean(np.exp(1j * np.angle(strongest[f"z{h}"].to_numpy()))))
        shots.loc[group.index, f"ref_phase{h}_deg"] = np.rad2deg(phase)
        shots.loc[group.index, f"signed{h}_mV"] = 1e3 * np.real(group[f"z{h}"].to_numpy() * np.exp(-1j * phase))
display(Markdown(f"**Scan variable:** `{scan}` with values {sorted(shots[scan].unique())}"))

METRICS = ["signed1_mV", "peak1_mV", "SNR1", "signed2_mV", "SNR2", "PMT_SNR", "MOT_atoms_mV",
           "PDA_V", "noise1_mV"]
summary = (shots.groupby(["acq", scan])[METRICS]
           .agg(["mean", "std", "sem", "count"]))
for acq, label in (("0", "Upward (_0)"), ("1", "Returning (_1)")):
    if acq in summary.index.get_level_values(0):
        display(Markdown(f"**{label}**"))
        table = summary.loc[acq][[(m, s) for m in ("signed1_mV", "SNR1", "SNR2", "PMT_SNR", "MOT_atoms_mV")
                                   for s in ("mean", "sem")] + [("SNR1", "count")]]
        display(table.round(3))

## Signal against the scan variable

Large markers with error bars are the mean ± standard error at each scan value; faint points are individual shots. Panels show the signed 1f and 2f signals, the 1f SNR, and two atom-number monitors: the PMT contrast and the MOT signal.

In [ ]:
# One figure per acquisition: signed signals, SNR, and atom monitors against the scan variable.
panels = [("signed1_mV", "signed 1f (mV)"), ("signed2_mV", "signed 2f (mV)"),
          ("SNR1", "1f SNR"), ("PMT_SNR", "PMT SNR"), ("MOT_atoms_mV", "MOT amp. (mV)"),
          ("PDA_V", "PDA DC (V)")]
for acq, name, label in (("0", "scan_upward", "upward"), ("1", "scan_returning", "returning")):
    group = shots[shots["acq"] == acq]
    if group.empty:
        continue
    stats = group.groupby(scan)
    fig, axes = plt.subplots(3, 2, sharex=True, layout="constrained")
    for ax, (column, ylabel) in zip(axes.flat, panels):
        ax.plot(group[scan], group[column], "o", color="0.6", alpha=0.35, ms=4, label="shots")
        mean, sem = stats[column].mean(), stats[column].sem()
        ax.errorbar(mean.index, mean.values, yerr=sem.values, fmt="o-", color="C0",
                    capsize=3, ms=6, label="mean ± s.e.")
        if column.startswith("signed"):
            ax.axhline(0, color="0.4", lw=0.8)
        if column == "SNR1":
            ax.axhline(2.4, color="C3", ls=":", label="noise-only level")
        ax.set_ylabel(ylabel, fontdict=label_font, labelpad=12)
    for ax in axes[-1]:
        ax.set_xlabel(scan, fontdict=label_font, labelpad=20)
    fig.suptitle(f"{label}: shots {FIRST_SHOT:04d}–{LAST_SHOT:04d}", **title_font)
    # Title on top, shared legend underneath so the two never collide.
    handles, labels = axes[1, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside lower center", ncols=3, frameon=False, prop=legend_font)
    show_figure(fig, name)

In [ ]:
# Export per-shot numbers and the grouped summary.
if SAVE_OUTPUTS:
    shots.drop(columns=["z1", "z2"]).assign(
        z1_real_mV=1e3 * np.real(shots["z1"]), z1_imag_mV=1e3 * np.imag(shots["z1"]),
        z2_real_mV=1e3 * np.real(shots["z2"]), z2_imag_mV=1e3 * np.imag(shots["z2"]),
    ).to_csv(OUTPUT_DIR / "per_shot.csv", index=False)
    summary.to_csv(OUTPUT_DIR / "summary_by_scan_value.csv")
    print(f"Saved per_shot.csv, summary_by_scan_value.csv and figures to {OUTPUT_DIR}")